In [1]:
import os, json, pickle, re
import numpy as np
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import os
import pandas as pd
import json
import pickle
from tqdm import tqdm
import sys
sys.path.append("..")
from shared_util.time_handler import TimeHandler

In [2]:
'''
datasets/
└── train/
    └── 2022-03-21/
        └── cloudbed-1/
            └── log/
                └── all/
                    ├── log_filebeat-testbed-log-service.csv
                    └── log_filebeat-testbed-log-envoy.csv
处理后的结构为
temp_data/raw_data/
└── train/
    └── 2022-03-21/
        └── cloudbed-1/
            └── raw_log/
                ├── log_patterns_count.csv
                └── istio_word_count.csv

Container 日志使用 Drain3 将日志行映射为 log template
统计每个 pod 在 每一分钟 内，每种 log pattern 出现了多少次
输出log_patterns_count.csv
================================================================================
log_patterns_count.csv
最后生成的横表表头为
timestamp {pod}; <log pattern {i} {pod}; <log pattern {i} ...
记录每分钟，每个pod对应template发生的次数
================================================================================



Istio / Envoy 日志对日志做正则清洗 + 分词
统计每个 pod、每个关键词在 每一分钟 内出现了多少次同时统计全局 total
输出istio_word_count.csv
================================================================================
istio_word_count.csv
最后生成的横表表头为
timestamp total keyword1 keyword2 ... {pod}; {w} {pod}; {w} ...
记录每分钟
总计出现的关键词
每个关键词出现的次数
每个pod上出现某个关键词的次数
================================================================================
将日志解析为按时间桶组织，每个pod实体出现关键词/模板的频率
'''

'\ndatasets/\n└── train/\n    └── 2022-03-21/\n        └── cloudbed-1/\n            └── log/\n                └── all/\n                    ├── log_filebeat-testbed-log-service.csv\n                    └── log_filebeat-testbed-log-envoy.csv\n处理后的结构为\ntemp_data/raw_data/\n└── train/\n    └── 2022-03-21/\n        └── cloudbed-1/\n            └── raw_log/\n                ├── log_patterns_count.csv\n                └── istio_word_count.csv\n\nContainer 日志使用 Drain3 将日志行映射为 log template\n统计每个 pod 在 每一分钟 内，每种 log pattern 出现了多少次\n输出log_patterns_count.csv\n================================================================================\nlog_patterns_count.csv\n最后生成的横表表头为\ntimestamp {pod}; <log pattern {i} {pod}; <log pattern {i} ...\n记录每分钟，每个pod对应template发生的次数\n================================================================================\n\n\n\nIstio / Envoy 日志对日志做正则清洗 + 分词\n统计每个 pod、每个关键词在 每一分钟 内出现了多少次同时统计全局 total\n输出istio_word_count.csv\n===================================================

In [3]:
import os, json, pickle, re
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig


# =========================
# 0) 基础配置（你已有）
# =========================
service_order_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

pod_list = [
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

service_language_dict = {
    "adservice": "Java",
    "cartservice": "C#",
    "checkoutservice": "Go",
    "currencyservice": "Node.js",
    "emailservice": "Python",
    "frontend": "Go",
    "paymentservice": "Node.js",
    "productcatalogservice": "Go",
    "recommendationservice": "Python",
    "shippingservice": "Go",
}

# 例：pod -> service
def rename_pod2service(pod_name: str) -> str:
    return pod_name.replace("2-0", "").replace("-0", "").replace("-1", "").replace("-2", "")

# 例：service -> pods（这里按你 aiops 的 pod_list 规则做映射）
def rename_service2pod(service: str):
    # 仅返回 -1/-2（与 pod_list 一致）
    return [f"{service}-0", f"{service}-1", f"{service}-2", f"{service}2-0"]


# =========================
# 1) 路径解析：dataset_type/date/cloudbed
# =========================
def parse_aiops_ids(log_path: str, data_base: str):
    """
    你的数据路径形如：
    {data_base}/{dataset_type}/{date}/{cloudbed}/log/all/log_filebeat-testbed-log-service.csv
    """
    rel = os.path.relpath(log_path, data_base)
    parts = rel.split(os.sep)

    # parts:
    # [dataset_type, date, cloudbed, 'log', 'all', filename]
    if len(parts) < 6:
        raise ValueError(f"Unexpected path structure: {rel}")

    dataset_type, date, cloudbed = parts[0], parts[1], parts[2]
    return dataset_type, date, cloudbed


# =========================
# 2) 时间桶：完全复刻原逻辑（24h，每分钟）
# =========================
def build_day_minute_timestamps(date: str, TimeHandler):
    start_ts = TimeHandler.datetime_to_timestamp(f"{date} 00:00:00")
    return [start_ts + i * 60 for i in range(0, 24 * 60)]


# =========================
# 3) Istio 分词（独立版，替代 self.calculate_word_count）
# =========================
def tokenize_istio_words(value: str, stopwords_regex_list):
    new_str = value.strip()
    for rg in stopwords_regex_list:
        new_str = re.sub(rg, "", new_str)
    words = [w for w in new_str.split(" ") if w]
    # 这里返回计数字典（复刻原逻辑会用 count）
    word_dict = {}
    for w in words:
        word_dict[w] = word_dict.get(w, 0) + 1
    return word_dict


# =========================
# 4) container log 计数（复刻原逻辑：Drain3 -> template_mined -> pattern index）
# =========================
def process_container_logs(
    timestamp_list: list,
    data_base_path: str,
    result_base_path: str,
    log_pattern_dict: dict,      # {lang: [pattern...]}
    miners: dict,                # {lang: TemplateMiner}
    all_entity_list: list,
    service_order_list: list,
    service_language_dict: dict,
    rename_pod2service,
    rename_service2pod,
    chunksize: int = 100_000,
):
    os.makedirs(result_base_path, exist_ok=True)

    # pattern -> index，加速
    '''
        log_pattern_dict={
            "Java":    [template_1, template_2, ...],
            "Go":      [...],
            "Python":  [...],
            ...
        }
        
        pattern_index[lang][temlate]=index
    '''
    pattern_index = {
        lang: {p: i for i, p in enumerate(plist)}
        for lang, plist in log_pattern_dict.items()
    }

    '''
        log_patterns_count.csv
        最后生成的横表表头为
        timestamp {pod}; <log pattern {i} {pod}; <log pattern {i} ...
        记录每分钟，每个pod对应template发生的次数
    '''
    result_dict = {"timestamp": timestamp_list}

    # 先建列：按 service_order -> service pods -> patterns
    for service in service_order_list:
        lang = service_language_dict[service]
        pods = rename_service2pod(service)
        # lang所对应模板列表的长度
        n_pat = len(log_pattern_dict.get(lang, []))
        for pod in pods:
            for i in range(n_pat):
                result_dict[f"{pod}; <log pattern {i}>"] = [0] * len(timestamp_list)
    '''
        result_dict={
            "timestamp": timestamp_list,
            "{pod}; <log pattern {i}>": List[0,...,] len=len(timestamp_list),
            每个pod对应其所有template
            记录每分钟，每个pod对应template发生的次数
        }
    '''
    reader = pd.read_csv(data_base_path, chunksize=chunksize)
    for chunk in tqdm(reader, desc="读取 chunk"):
        for row in chunk.itertuples(index=False):
            ts = getattr(row, "timestamp", None)
            cmdb_id = getattr(row, "cmdb_id", None)
            value = getattr(row, "value", None)

            if ts is None or cmdb_id is None or value is None:
                continue
            if cmdb_id not in all_entity_list:
                continue
            if not isinstance(value, str):
                continue

            index = int((ts - timestamp_list[0]) / 60)
            if index < 0 or index >= len(timestamp_list):
                continue

            service = rename_pod2service(cmdb_id)
            lang = service_language_dict.get(service)
            if lang is None:
                continue

            template = miners[lang].add_log_message(value).get("template_mined", "")
            if not template:
                continue

            pat_i = pattern_index.get(lang, {}).get(template)
            if pat_i is None:
                continue

            col = f"{cmdb_id}; <log pattern {pat_i}>"
            if col in result_dict:
                result_dict[col][index] += 1
    pd.DataFrame(result_dict).to_csv(os.path.join(result_base_path, "log_patterns_count.csv"), index=False)


# =========================
# 5) istio log 计数（复刻原逻辑：word count + total + pod;word）
# =========================
def process_istio_logs(
    timestamp_list: list,
    data_base_path: str,
    result_base_path: str,
    istio_words: list,
    stopwords_regex_list: list,
    all_entity_list: list,
    pod_list: list,
    chunksize: int = 100_000,
):
    os.makedirs(result_base_path, exist_ok=True)
    '''
        istio_word_count.csv
        最后生成的横表表头为
        timestamp total keyword1 keyword2 ... {pod}; {w} {pod}; {w} ...
        记录每分钟
        总计出现的关键词
        每个关键词出现的次数
        每个pod上出现某个关键词的次数
    '''
    istio_set = set(istio_words)

    result_dict = {
        "timestamp": timestamp_list,
        "total": [0] * len(timestamp_list),
    }

    valid_words = [w for w in istio_words if ("latency=" not in w and "ttl=" not in w)]
    # 从关键词表中剔除包含"latency="和"ttl="的词
    for w in valid_words:
        result_dict[w] = [0] * len(timestamp_list)


    # pod;word 列（只给 pod_list）
    for pod in pod_list:
        for w in valid_words:
            result_dict[f"{pod}; {w}"] = [0] * len(timestamp_list)

    '''
        result_dict={
            "timestamp": timestamp_list,
            "total": List[0,...,] len=len(timestamp_list),
            "valid_word": List[0,...,] len=len(timestamp_list),
            "{pod}; {w}": List[0,...,] len=len(timestamp_list),
            每个pod对应关键词
            记录每分钟，每个pod对应关键词发生的次数
        }
    '''
    reader = pd.read_csv(data_base_path, chunksize=chunksize)
    for chunk in tqdm(reader, desc="读取 chunk"):
        for row in chunk.itertuples(index=False):
            ts = getattr(row, "timestamp", None)
            cmdb_id = getattr(row, "cmdb_id", None)
            value = getattr(row, "value", None)

            if ts is None or cmdb_id is None or value is None:
                continue
            if cmdb_id not in all_entity_list:
                continue
            if cmdb_id not in pod_list:  # 复刻原逻辑：必须在 pod_order
                continue
            if not isinstance(value, str):
                continue

            index = int((ts - timestamp_list[0]) / 60)
            if index < 0 or index >= len(timestamp_list):
                continue

            word_dict = tokenize_istio_words(value, stopwords_regex_list)

            for w, count in word_dict.items():
                if w not in istio_set or "latency=" in w or "ttl=" in w:
                    continue
                result_dict["total"][index] += count
                result_dict[w][index] += count
                result_dict[f"{cmdb_id}; {w}"][index] += count

                
    pd.DataFrame(result_dict).to_csv(os.path.join(result_base_path, "istio_word_count.csv"), index=False)


# =========================
# 6) 总控：extract_log_features（按目录镜像输出结构）
# =========================
def extract_log_features(
    data_base: str,
    patterns_json: str,
    istio_words_json: str,
    miner_pkl_path: str,
    result_base_path: str,
    TimeHandler,                      # 你项目里的 TimeHandler（外部注入）
    all_entity_list: list,
    stopwords_regex_list: list,
    service_language_dict,
    target_service="log_filebeat-testbed-log-service.csv",
    target_envoy="log_filebeat-testbed-log-envoy.csv",
    chunksize: int = 100_000,
    skip_if_path_contains=("test_data",),   # 你想跳过就保持
):
    # load patterns / words / miners
    with open(patterns_json, "r", encoding="utf-8") as f:
        log_pattern_dict = json.load(f)
    with open(istio_words_json, "r", encoding="utf-8") as f:
        istio_words = json.load(f)["word_set"]
    # 2) 自动得到语言集合（适配不同数据集）
    languages = sorted(set(service_language_dict.values()))

    # 3) 为每种语言创建 miner + pattern 容器
    with open(miner_pkl_path, "rb") as f:
        miners = pickle.load(f)


    '''
        log_pattern_dict={
            "Java":    [template_1, template_2, ...],
            "Go":      [...],
            "Python":  [...],
            ...
        }

        miners={
            language: TemplateMiner instance
        }

        istio_words=[] 关键词表
    '''

        
    for root, _, files in os.walk(data_base):
        if target_service not in files:
            continue

        service_path = os.path.join(root, target_service)

        # 跳过 test_data（你的策略）
        # if any(x in service_path for x in skip_if_path_contains):
        #     continue

        dataset_type, date, cloudbed = parse_aiops_ids(service_path, data_base)

        # 完全复刻：固定 24h 时间桶（分钟粒度）
        timestamp_list = build_day_minute_timestamps(date, TimeHandler)

        # 输出路径镜像：result_base_path/dataset_type/date/cloudbed/raw_log
        out_dir = os.path.join(result_base_path, dataset_type, date, cloudbed, "raw_log")

        print(f"[INFO] container: {dataset_type} {date} {cloudbed}")
        process_container_logs(
            timestamp_list=timestamp_list,
            data_base_path=service_path,
            result_base_path=out_dir,
            log_pattern_dict=log_pattern_dict,
            miners=miners,
            all_entity_list=all_entity_list,
            service_order_list=service_order_list,
            service_language_dict=service_language_dict,
            rename_pod2service=rename_pod2service,
            rename_service2pod=rename_service2pod,
            chunksize=chunksize,
        )

        # 同目录下 envoy
        envoy_path = os.path.join(root, target_envoy)
        if os.path.exists(envoy_path):
            print(f"[INFO] istio: {dataset_type} {date} {cloudbed}")
            process_istio_logs(
                timestamp_list=timestamp_list,
                data_base_path=envoy_path,
                result_base_path=out_dir,
                istio_words=istio_words,
                stopwords_regex_list=stopwords_regex_list,
                all_entity_list=all_entity_list,
                pod_list=pod_list,
                chunksize=chunksize,
            )

In [4]:
all_entity_list = [
    'node-1', 
    'node-2', 
    'node-3', 
    'node-4', 
    'node-5', 
    'node-6', 
    'adservice', 
    'cartservice', 
    'checkoutservice', 
    'currencyservice', 
    'emailservice', 
    'frontend', 
    'paymentservice', 
    'productcatalogservice', 
    'recommendationservice', 
    'shippingservice', 
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [5]:
stopwords_regex_list = [
        " \"*([0-9]{1,3}\\.[0-9]{1,3}\\.[0-9]{1,3}\\.[0-9]{1,3})\\:?([0-9]{1,5})?\"*",
        "\"(\\w|\\d)*-(\\w|\\d)*-(\\w|\\d)*-(\\w|\\d)*-(\\w|\\d)*\"",
        "( |^)\"*(GET|POST|PUT|DELETE)\"*",
        " \"*[0-9]+\"*",
        " (\"*grpc-(\\w)*/)[0-9]+.[0-9]+.[0-9]+\"*",
        " (\"*Go(-(\\w)*)*/)[0-9]+(.[0-9]+)*\"*",
        "( |^)\"*-\"*",
        " (default|(\\(manylinux; chttp2\\))|(HTTP/\\d(.\\d)*\"*))\"*"
    ]

In [6]:
extract_log_features(
    data_base="/home/ZZData/Eadro/preprocess/datasets/aiops2022/",
    patterns_json="../../temp_data/aiops22/analysis/log/log_patterns.json",
    istio_words_json="../../temp_data/aiops22/analysis/log/istio_words.json",
    miner_pkl_path="../../temp_data/aiops22/analysis/log/log_template_miner.pkl",
    result_base_path="../../temp_data/aiops22/raw_data",
    TimeHandler=TimeHandler,
    service_language_dict=service_language_dict,
    all_entity_list=all_entity_list,
    stopwords_regex_list=stopwords_regex_list,
    skip_if_path_contains=("test_data",),  # 你要处理 test_data 就把这行去掉
)

[INFO] container: test_data 2022-05-03 cloudbed


读取 chunk: 0it [00:00, ?it/s]

total          : took   106.69 s (100.00%), 10,282,256 samples,   10.38 ms / 1000 samples,       96,376.30 hz
drain          : took    55.48 s ( 52.00%), 10,282,256 samples,    5.40 ms / 1000 samples,      185,345.66 hz
mask           : took    31.45 s ( 29.48%), 10,282,256 samples,    3.06 ms / 1000 samples,      326,951.65 hz
tree_search    : took    25.81 s ( 24.20%), 10,282,256 samples,    2.51 ms / 1000 samples,      398,322.10 hz
cluster_exist  : took    12.14 s ( 11.38%), 10,282,238 samples,    1.18 ms / 1000 samples,      847,150.76 hz
create_cluster : took     0.00 s (  0.00%),         18 samples,   10.15 ms / 1000 samples,       98,560.67 hz
total          : took   292.88 s (100.00%), 23,495,276 samples,   12.47 ms / 1000 samples,       80,220.92 hz
drain          : took   143.05 s ( 48.84%), 23,495,276 samples,    6.09 ms / 1000 samples,      164,242.88 hz
mask           : took   104.47 s ( 35.67%), 23,495,276 samples,    4.45 ms / 1000 samples,      224,895.43 hz
tree_searc

读取 chunk: 37it [00:58,  1.59s/it]

total          : took   113.90 s (100.00%), 10,937,423 samples,   10.41 ms / 1000 samples,       96,024.89 hz
drain          : took    59.36 s ( 52.12%), 10,937,423 samples,    5.43 ms / 1000 samples,      184,246.20 hz
mask           : took    33.48 s ( 29.39%), 10,937,423 samples,    3.06 ms / 1000 samples,      326,723.89 hz
tree_search    : took    27.71 s ( 24.33%), 10,937,423 samples,    2.53 ms / 1000 samples,      394,639.24 hz
cluster_exist  : took    12.96 s ( 11.38%), 10,937,405 samples,    1.19 ms / 1000 samples,      843,825.58 hz
create_cluster : took     0.00 s (  0.00%),         18 samples,   10.15 ms / 1000 samples,       98,560.67 hz
total          : took   312.48 s (100.00%), 24,937,914 samples,   12.53 ms / 1000 samples,       79,805.52 hz
drain          : took   153.31 s ( 49.06%), 24,937,914 samples,    6.15 ms / 1000 samples,      162,658.80 hz
mask           : took   110.97 s ( 35.51%), 24,937,914 samples,    4.45 ms / 1000 samples,      224,735.18 hz
tree_searc

...

读取 chunk: 37it [00:59,  1.62s/it]

total          : took   352.82 s (100.00%), 35,857,283 samples,    9.84 ms / 1000 samples,      101,630.56 hz
drain          : took   159.09 s ( 45.09%), 35,857,283 samples,    4.44 ms / 1000 samples,      225,385.21 hz
mask           : took   122.89 s ( 34.83%), 35,857,283 samples,    3.43 ms / 1000 samples,      291,780.91 hz
tree_search    : took    54.11 s ( 15.34%), 35,857,283 samples,    1.51 ms / 1000 samples,      662,734.71 hz
cluster_exist  : took    43.29 s ( 12.27%), 35,857,277 samples,    1.21 ms / 1000 samples,      828,259.60 hz
create_cluster : took     0.00 s (  0.00%),          6 samples,    9.70 ms / 1000 samples,      103,138.62 hz
total          : took   273.37 s (100.00%), 25,371,034 samples,   10.77 ms / 1000 samples,       92,809.36 hz
drain          : took   145.86 s ( 53.36%), 25,371,034 samples,    5.75 ms / 1000 samples,      173,937.54 hz
mask           : took    77.97 s ( 28.52%), 25,371,034 samples,    3.07 ms / 1000 samples,      325,409.28 hz
tree_searc

读取 chunk: 43it [01:08,  1.59s/it]


[INFO] istio: training_data_with_faults 2022-03-24 cloudbed-3


读取 chunk: 74it [02:38,  2.14s/it]
